# 00_02: Transformer Architecture

**Learning Objectives:**
- Understand the core components of the Transformer architecture
- Explore self-attention and multi-head attention mechanisms
- Compare encoder-only, decoder-only, and encoder-decoder architectures
- Inspect real model weights, layers, and attention patterns using HuggingFace
- Visualize attention heads to see what the model "looks at"

**Prerequisites:** Notebook 00_01 (Tokenization & Embeddings). Basic linear algebra helpful but not required.

## Prerequisites

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 8GB recommended |
| **Storage** | ~1.5GB for model downloads |
| **GPU** | Not required |
| **Models** | bert-base-uncased (~440MB), distilgpt2 (~82MB) |

## Expected Behaviors

### What You'll See
- Transformer models are stacks of layers, each containing attention and feed-forward blocks
- Attention weights show which tokens attend to which other tokens
- BERT (encoder) produces bidirectional representations; GPT-2 (decoder) produces left-to-right
- Model parameter counts match published specifications

### First Run
- Model downloads: ~500MB total (cached after first run)
- All computations run on CPU in seconds

### Common Observations
- [CLS] token often attends broadly across the whole sentence
- Punctuation tokens frequently receive high attention
- Different attention heads learn different patterns (positional, syntactic, semantic)

## Overview

The **Transformer** architecture, introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al., is the foundation of virtually every modern language model. Rather than processing text sequentially like RNNs, Transformers use **self-attention** to relate every token to every other token in parallel.

This notebook takes a hands-on approach: instead of just reading about Transformers, we'll load real HuggingFace models and inspect their internals — weights, layers, attention patterns, and outputs. By the end, you'll have an intuitive understanding of what happens when text passes through a Transformer.

### Three Transformer Families

| Family | Architecture | Example Models | Use Cases |
|--------|-------------|----------------|----------|
| **Encoder-only** | Bidirectional self-attention | BERT, RoBERTa, DistilBERT | Classification, NER, QA |
| **Decoder-only** | Causal (left-to-right) self-attention | GPT-2, LLaMA, Mistral | Text generation |
| **Encoder-Decoder** | Cross-attention between encoder and decoder | T5, BART, MarianMT | Translation, summarization |

## Setup and Installation

We'll use BERT as our encoder-only example and DistilGPT2 as our decoder-only example. Both are small enough to inspect comfortably on CPU.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import random
import transformers
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
)
import matplotlib.pyplot as plt
import pandas as pd
import sys

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

We load two models to contrast the two main Transformer families. Both are small and CPU-friendly.

In [ ]:
# Encoder-only model (bidirectional)
ENCODER_MODEL = 'bert-base-uncased'        # 110M params, ~440MB

# Decoder-only model (causal / left-to-right)
DECODER_MODEL = 'distilgpt2'                # 82M params, ~82MB

# Option: Larger models for GPU users
# ENCODER_MODEL = 'bert-large-uncased'      # 340M params, ~1.3GB
# DECODER_MODEL = 'gpt2-medium'             # 345M params, ~1.5GB

## Part 1: Inspecting Model Architecture

Every HuggingFace model is a PyTorch `nn.Module`. We can print the full architecture to see every layer, its type, and its dimensions. This is the best way to build intuition about what's inside a Transformer.

In [ ]:
# Load the encoder model
print('Loading encoder model...')
encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL)
encoder_model = AutoModel.from_pretrained(ENCODER_MODEL).to(device)
encoder_model.eval()

print(f'\nModel: {ENCODER_MODEL}')
print(f'Type: Encoder-only (bidirectional)')
print(encoder_model)

Notice the repeating structure: BERT has 12 identical "BertLayer" blocks stacked on top of each other. Each block contains:

1. **BertAttention** — Multi-head self-attention (query, key, value projections)
2. **BertIntermediate** — Feed-forward expansion (hidden_size → intermediate_size)
3. **BertOutput** — Feed-forward projection back + residual connection + LayerNorm

This is the fundamental building block of every Transformer. Let's count the parameters to verify the published numbers.

In [ ]:
def count_parameters(model: torch.nn.Module) -> pd.DataFrame:
    """Count trainable parameters per top-level module.

    Args:
        model: A PyTorch model.

    Returns:
        DataFrame with module names, parameter counts, and percentages.
    """
    data: list[dict[str, object]] = []
    total = sum(p.numel() for p in model.parameters())
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        data.append({
            'Module': name,
            'Parameters': f'{params:,}',
            'Percentage': f'{params / total:.1%}',
        })
    data.append({
        'Module': 'TOTAL',
        'Parameters': f'{total:,}',
        'Percentage': '100.0%',
    })
    return pd.DataFrame(data)


print(f'=== {ENCODER_MODEL} ===')
count_parameters(encoder_model)

The embeddings layer is typically a small fraction of the total; the bulk of the parameters live in the stacked attention + feed-forward layers. This is why deeper models (more layers) or wider models (larger hidden size) grow so quickly in parameter count.

## Part 2: Self-Attention — The Core Mechanism

Self-attention lets each token compute a weighted sum over all other tokens in the sequence. For each token, we compute:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

Where:
- $\mathbf{Q}$ (Query): "What am I looking for?"
- $\mathbf{K}$ (Key): "What do I contain?"
- $\mathbf{V}$ (Value): "What information do I provide?"
- $d_k$: Dimension of the key vectors (scaling factor to stabilize gradients)

Let's implement this from scratch and then verify it matches the model's output.

In [ ]:
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention from scratch.

    Args:
        query: Query tensor of shape (batch, seq_len, d_k).
        key: Key tensor of shape (batch, seq_len, d_k).
        value: Value tensor of shape (batch, seq_len, d_v).
        mask: Optional mask tensor. Positions with True are masked (ignored).

    Returns:
        Tuple of (output, attention_weights).
    """
    d_k = query.size(-1)
    # Step 1: Compute attention scores
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    
    # Step 2: Apply mask (for causal / padding)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    
    # Step 3: Softmax to get weights
    attention_weights = torch.softmax(scores, dim=-1)
    
    # Step 4: Weighted sum of values
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights


# Demo with a small example
torch.manual_seed(SEED)
seq_len = 4
d_k = 8
query = torch.randn(1, seq_len, d_k)
key = torch.randn(1, seq_len, d_k)
value = torch.randn(1, seq_len, d_k)

output, weights = scaled_dot_product_attention(query, key, value)
print('Attention weights (each row sums to 1.0):')
print(weights.squeeze().numpy().round(3))
print(f'\nRow sums: {weights.squeeze().sum(dim=-1).numpy()}')
print(f'Output shape: {output.shape}  (same as input)')

Each row of the attention weights matrix represents one token's "attention distribution" over all tokens. The weights sum to 1.0 because of the softmax. Higher weights mean that token is paying more attention to the corresponding position.

### Causal Masking (Decoder-Only)

In decoder models like GPT-2, each token can only attend to tokens that came **before** it (left-to-right). This is enforced with a causal mask that sets future positions to $-\infty$ before the softmax.

In [ ]:
# Create a causal mask (True = masked / blocked)
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
print('Causal mask (True = blocked):')
print(causal_mask.int().numpy())

# Apply causal attention
output_causal, weights_causal = scaled_dot_product_attention(
    query, key, value, mask=causal_mask.unsqueeze(0)
)
print('\nCausal attention weights (upper triangle is zero):')
print(weights_causal.squeeze().numpy().round(3))

Notice how the upper triangle is now zero — token 0 can only see itself, token 1 can see tokens 0-1, and so on. This is what makes decoder models **autoregressive**: they generate one token at a time, conditioned only on previous tokens.

## Part 3: Multi-Head Attention

Instead of computing a single attention pattern, Transformers compute **multiple** attention patterns in parallel ("heads"). Each head can learn a different type of relationship:

- Head 1 might learn syntactic dependencies (subject-verb agreement)
- Head 2 might learn positional patterns (attend to adjacent words)
- Head 3 might learn semantic relationships (word meaning connections)

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}^O$$

Let's extract the actual multi-head attention patterns from our BERT model.

In [ ]:
# Run a sentence through BERT and extract attention
text = 'The cat sat on the mat because it was tired'
inputs = encoder_tokenizer(text, return_tensors='pt').to(device)
tokens = encoder_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = encoder_model(**inputs, output_attentions=True)

# outputs.attentions is a tuple: one tensor per layer
# Each tensor has shape: (batch, num_heads, seq_len, seq_len)
print(f'Number of layers: {len(outputs.attentions)}')
print(f'Attention shape per layer: {outputs.attentions[0].shape}')
print(f'Tokens: {tokens}')

config = encoder_model.config
print(f'\nBERT config: {config.num_hidden_layers} layers, '
      f'{config.num_attention_heads} heads, '
      f'hidden_size={config.hidden_size}, '
      f'd_k={config.hidden_size // config.num_attention_heads}')

## Part 4: Visualizing Attention Patterns

Visualizing attention is one of the best ways to understand what the model is doing. We'll look at attention patterns across different layers and heads to see how information flows through the network.

In [ ]:
def plot_attention_heads(
    attentions: tuple[torch.Tensor, ...],
    tokens: list[str],
    layer: int,
    heads: list[int] | None = None,
) -> None:
    """Visualize attention weight heatmaps for selected heads in a layer.

    Args:
        attentions: Tuple of attention tensors from model output.
        tokens: List of token strings.
        layer: Layer index to visualize.
        heads: List of head indices. Defaults to first 4 heads.
    """
    if heads is None:
        heads = list(range(min(4, attentions[layer].shape[1])))
    
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 5))
    if len(heads) == 1:
        axes = [axes]
    
    for idx, head in enumerate(heads):
        attn = attentions[layer][0, head].cpu().numpy()
        axes[idx].imshow(attn, cmap='viridis', vmin=0, vmax=1)
        axes[idx].set_xticks(range(len(tokens)))
        axes[idx].set_yticks(range(len(tokens)))
        axes[idx].set_xticklabels(tokens, rotation=90, fontsize=8)
        axes[idx].set_yticklabels(tokens, fontsize=8)
        axes[idx].set_title(f'Layer {layer}, Head {head}', fontsize=10)
        axes[idx].set_xlabel('Key (attended to)')
        axes[idx].set_ylabel('Query (attending from)')
    
    plt.suptitle(f'Attention Patterns — Layer {layer}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# Visualize early layer (often positional / local patterns)
plot_attention_heads(outputs.attentions, tokens, layer=0, heads=[0, 1, 2, 3])

In [ ]:
# Visualize a middle layer (more semantic patterns)
plot_attention_heads(outputs.attentions, tokens, layer=6, heads=[0, 1, 2, 3])

In [ ]:
# Visualize last layer (task-relevant patterns)
plot_attention_heads(outputs.attentions, tokens, layer=11, heads=[0, 1, 2, 3])

### What to Look For

Across these visualizations, you may notice several common patterns:

- **Diagonal patterns**: Tokens attend to themselves or adjacent tokens (local/positional)
- **Vertical stripes**: Many tokens attend to special tokens like [CLS] or [SEP]
- **Off-diagonal structure**: Tokens attend to specific other tokens (syntactic or semantic links)
- **Uniform attention**: Some heads attend roughly equally everywhere (these may seem "unused" but still contribute)

Earlier layers tend to capture local/positional patterns, while deeper layers capture more abstract semantic relationships. This is a general trend across Transformer models.

## Part 5: Encoder vs Decoder Comparison

Let's now load a decoder model and compare the internal structure, attention patterns, and behavior. The key difference: decoder models use **causal masking** so each token can only attend to previous tokens.

In [ ]:
# Load decoder model
print('Loading decoder model...')
decoder_tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)
decoder_model = AutoModelForCausalLM.from_pretrained(DECODER_MODEL).to(device)
decoder_model.eval()

print(f'\nModel: {DECODER_MODEL}')
print(f'Type: Decoder-only (causal)')

# Compare architectures side by side
encoder_config = encoder_model.config
decoder_config = decoder_model.config

comparison = pd.DataFrame({
    'Property': [
        'Architecture', 'Layers', 'Attention Heads',
        'Hidden Size', 'd_k (per head)', 'Feed-Forward Size',
        'Vocab Size', 'Max Position', 'Total Parameters',
        'Attention Type',
    ],
    ENCODER_MODEL: [
        'Encoder-only', encoder_config.num_hidden_layers,
        encoder_config.num_attention_heads, encoder_config.hidden_size,
        encoder_config.hidden_size // encoder_config.num_attention_heads,
        encoder_config.intermediate_size, encoder_config.vocab_size,
        encoder_config.max_position_embeddings,
        f'{sum(p.numel() for p in encoder_model.parameters()):,}',
        'Bidirectional',
    ],
    DECODER_MODEL: [
        'Decoder-only', decoder_config.n_layer,
        decoder_config.n_head, decoder_config.n_embd,
        decoder_config.n_embd // decoder_config.n_head,
        decoder_config.n_inner if decoder_config.n_inner else decoder_config.n_embd * 4,
        decoder_config.vocab_size,
        decoder_config.n_positions,
        f'{sum(p.numel() for p in decoder_model.parameters()):,}',
        'Causal (masked)',
    ],
})
comparison

The table above reveals the key structural differences. BERT uses bidirectional attention (every token sees the full context), while DistilGPT2 uses causal attention (each token only sees tokens to its left). This architectural choice determines what each model is good at:

- **Bidirectional** (BERT): Excellent for understanding tasks (classification, NER, QA) because it sees full context
- **Causal** (GPT-2): Excellent for generation tasks because it naturally predicts the next token

In [ ]:
# Visualize decoder attention (causal pattern)
decoder_text = 'The cat sat on the mat'
decoder_inputs = decoder_tokenizer(decoder_text, return_tensors='pt').to(device)
decoder_tokens = decoder_tokenizer.convert_ids_to_tokens(
    decoder_inputs['input_ids'][0]
)

with torch.no_grad():
    decoder_outputs = decoder_model(
        **decoder_inputs, output_attentions=True
    )

print(f'Decoder tokens: {decoder_tokens}')
print(f'Number of layers: {len(decoder_outputs.attentions)}')

# Show causal attention pattern — notice the triangular mask
plot_attention_heads(
    decoder_outputs.attentions, decoder_tokens,
    layer=0, heads=[0, 1, 2, 3]
)

Notice the clear triangular pattern in the decoder attention: the upper-right portion is always zero because each token cannot attend to future tokens. Compare this to the BERT attention plots above, where the full matrix can have non-zero values.

## Part 6: How Representations Evolve Through Layers

As input passes through each Transformer layer, the representations become progressively more abstract. Early layers capture surface-level features (syntax, word identity), while deeper layers capture semantic meaning. Let's verify this by comparing the similarity of token representations across layers.

In [ ]:
def track_representations(
    model: torch.nn.Module,
    tokenizer: transformers.PreTrainedTokenizer,
    text: str,
) -> tuple[list[torch.Tensor], list[str]]:
    """Extract hidden states from all layers of a model.

    Args:
        model: A HuggingFace model (encoder or decoder).
        tokenizer: Corresponding tokenizer.
        text: Input text to process.

    Returns:
        Tuple of (list of hidden state tensors, list of token strings).
    """
    inputs = tokenizer(text, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # hidden_states[0] = embedding output, [1] = after layer 1, etc.
    hidden_states = [h.squeeze(0).cpu() for h in outputs.hidden_states]
    return hidden_states, tokens


hidden_states, track_tokens = track_representations(
    encoder_model, encoder_tokenizer,
    'The bank by the river bank was closed'
)

print(f'Layers (including embedding): {len(hidden_states)}')
print(f'Hidden state shape per layer: {hidden_states[0].shape}')
print(f'Tokens: {track_tokens}')

In [ ]:
def plot_token_similarity_across_layers(
    hidden_states: list[torch.Tensor],
    tokens: list[str],
    token_a_idx: int,
    token_b_idx: int,
) -> None:
    """Plot cosine similarity between two tokens across all layers.

    Args:
        hidden_states: List of hidden state tensors per layer.
        tokens: List of token strings.
        token_a_idx: Index of first token.
        token_b_idx: Index of second token.
    """
    similarities: list[float] = []
    for layer_hidden in hidden_states:
        vec_a = layer_hidden[token_a_idx]
        vec_b = layer_hidden[token_b_idx]
        cosine_sim = torch.nn.functional.cosine_similarity(
            vec_a.unsqueeze(0), vec_b.unsqueeze(0)
        ).item()
        similarities.append(cosine_sim)
    
    layers = list(range(len(similarities)))
    plt.figure(figsize=(10, 5))
    plt.plot(layers, similarities, 'o-', linewidth=2, markersize=6)
    plt.xlabel('Layer (0 = embedding)', fontsize=12)
    plt.ylabel('Cosine Similarity', fontsize=12)
    plt.title(
        f'Similarity between "{tokens[token_a_idx]}" (pos {token_a_idx}) '
        f'and "{tokens[token_b_idx]}" (pos {token_b_idx}) across layers',
        fontsize=12,
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Compare the two occurrences of "bank" in different contexts
# "The bank by the river bank was closed"
#   [CLS] the bank by the river bank was closed [SEP]
#    0     1   2    3  4    5     6    7    8     9
print(f'Tokens: {track_tokens}')
print('Comparing "bank" (pos 2, financial?) vs "bank" (pos 6, river)')
plot_token_similarity_across_layers(hidden_states, track_tokens, 2, 6)

This is one of the most powerful demonstrations of contextual embeddings. The two occurrences of "bank" start with **identical** representations at layer 0 (same token, same embedding). But as they pass through the layers, the model incorporates context: one "bank" is near "river", the other near "closed". Their representations **diverge** as the model disambiguates them.

This is fundamentally different from older word embedding approaches (Word2Vec, GloVe) where "bank" always had the same vector regardless of context.

## Part 7: The Feed-Forward Network

Each Transformer layer has two main components: attention (which we've explored) and a **feed-forward network** (FFN). The FFN is applied independently to each token position:

$$\text{FFN}(\mathbf{x}) = \text{GELU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

The FFN first expands the representation (hidden_size → 4×hidden_size), applies a nonlinearity, then projects back down. This expansion is where much of the model's "knowledge" is stored.

In [ ]:
# Inspect the FFN dimensions in BERT
layer_0 = encoder_model.encoder.layer[0]
ffn_up = layer_0.intermediate.dense
ffn_down = layer_0.output.dense

print('Feed-Forward Network in BERT Layer 0:')
print(f'  Up projection:   {ffn_up.in_features} → {ffn_up.out_features} '
      f'(expansion ratio: {ffn_up.out_features / ffn_up.in_features:.0f}x)')
print(f'  Down projection: {ffn_down.in_features} → {ffn_down.out_features}')
print(f'  Activation: {encoder_model.config.hidden_act}')
print(f'\n  FFN params per layer: '
      f'{ffn_up.in_features * ffn_up.out_features + ffn_down.in_features * ffn_down.out_features:,}')
print(f'  Attention params per layer: '
      f'{sum(p.numel() for p in layer_0.attention.parameters()):,}')

An important insight: the feed-forward network actually contains **more parameters** than the attention mechanism in each layer. While attention gets most of the attention (pun intended) in explanations, the FFN is where much of the model's learned knowledge resides. Research has shown that individual FFN neurons can encode specific facts.

## Part 8: Layer Normalization and Residual Connections

Two critical components that make Transformers trainable are **residual connections** (skip connections) and **layer normalization**. Without them, deep Transformers would be nearly impossible to train.

The residual connection adds the input back to the output of each sub-layer:
$$\mathbf{x}_{\text{out}} = \text{LayerNorm}(\mathbf{x}_{\text{in}} + \text{SubLayer}(\mathbf{x}_{\text{in}}))$$

This means information can flow directly through the network even if some layers contribute little, making training much more stable.

In [ ]:
# Inspect LayerNorm parameters
layer_norm = encoder_model.encoder.layer[0].attention.output.LayerNorm
print('LayerNorm after attention in Layer 0:')
print(f'  Normalized shape: {layer_norm.normalized_shape}')
print(f'  Gamma (scale) stats: mean={layer_norm.weight.mean():.4f}, '
      f'std={layer_norm.weight.std():.4f}')
print(f'  Beta (shift) stats: mean={layer_norm.bias.mean():.4f}, '
      f'std={layer_norm.bias.std():.4f}')

# Demonstrate residual connection effect
print('\n--- Residual Connection Demo ---')
sample_input = hidden_states[0][1:2]  # Take one token embedding
layer_output = hidden_states[1][1:2]   # After first layer

# Check if output is close to input (residual allows gradual refinement)
cosine_sim = torch.nn.functional.cosine_similarity(
    sample_input, layer_output
).item()
print(f'Cosine similarity between input and layer-1 output: {cosine_sim:.4f}')
print('(High similarity confirms residual connections preserve information)')

## Part 9: Encoder-Decoder Architecture

The third Transformer family combines both: the **encoder** processes the input, and the **decoder** generates the output while attending to the encoder's representations via **cross-attention**. Let's inspect a T5 model configuration to understand this.

In [ ]:
# Load T5 config (no need to download the full model)
t5_config = AutoConfig.from_pretrained('t5-small')

architecture_summary = pd.DataFrame({
    'Property': [
        'Family', 'Attention Type', 'Best For',
        'Example', 'Layers', 'Hidden Size', 'Heads',
    ],
    'Encoder-Only': [
        'BERT family', 'Bidirectional', 'Understanding (classify, extract)',
        ENCODER_MODEL, encoder_config.num_hidden_layers,
        encoder_config.hidden_size, encoder_config.num_attention_heads,
    ],
    'Decoder-Only': [
        'GPT family', 'Causal (left→right)', 'Generation (write, chat)',
        DECODER_MODEL, decoder_config.n_layer,
        decoder_config.n_embd, decoder_config.n_head,
    ],
    'Encoder-Decoder': [
        'T5 family', 'Bidirectional + Causal + Cross', 'Seq2Seq (translate, summarize)',
        't5-small', t5_config.num_layers,
        t5_config.d_model, t5_config.num_heads,
    ],
})
architecture_summary

### Choosing the Right Architecture

When selecting a model for a task, the architecture matters:

- **Need to understand text?** (sentiment, NER, similarity) → Encoder-only (BERT, RoBERTa)
- **Need to generate text?** (chatbot, code, creative writing) → Decoder-only (GPT-2, LLaMA)
- **Need to transform text?** (translation, summarization) → Encoder-decoder (T5, BART) or decoder-only

In practice, large decoder-only models (GPT-4, Claude, LLaMA) have become dominant for many tasks due to their scaling properties. But for efficient, task-specific models, encoder-only architectures remain excellent.

## Performance Benchmarking

Let's measure the inference time for our models to understand the computational cost of different architectures.

In [ ]:
import time


def benchmark_model(
    model: torch.nn.Module,
    tokenizer: transformers.PreTrainedTokenizer,
    text: str,
    num_runs: int = 10,
) -> dict[str, float]:
    """Benchmark model forward pass latency.

    Args:
        model: A HuggingFace model.
        tokenizer: Corresponding tokenizer.
        text: Input text to benchmark with.
        num_runs: Number of inference runs for averaging.

    Returns:
        Dictionary with timing statistics.
    """
    inputs = tokenizer(text, return_tensors='pt').to(device)
    
    # Warmup
    with torch.no_grad():
        model(**inputs)
    
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.no_grad():
            model(**inputs)
        times.append(time.perf_counter() - start)
    
    return {
        'mean_ms': np.mean(times) * 1000,
        'std_ms': np.std(times) * 1000,
        'min_ms': np.min(times) * 1000,
    }


bench_text = 'The Transformer architecture has revolutionized natural language processing.'

encoder_bench = benchmark_model(encoder_model, encoder_tokenizer, bench_text)
decoder_bench = benchmark_model(decoder_model, decoder_tokenizer, bench_text)

bench_df = pd.DataFrame({
    'Model': [ENCODER_MODEL, DECODER_MODEL],
    'Parameters': [
        f'{sum(p.numel() for p in encoder_model.parameters()) / 1e6:.0f}M',
        f'{sum(p.numel() for p in decoder_model.parameters()) / 1e6:.0f}M',
    ],
    'Mean Latency (ms)': [f'{encoder_bench["mean_ms"]:.1f}', f'{decoder_bench["mean_ms"]:.1f}'],
    'Std (ms)': [f'{encoder_bench["std_ms"]:.1f}', f'{decoder_bench["std_ms"]:.1f}'],
    'Device': [str(device)] * 2,
})
bench_df

## Exercises

1. **Pronoun resolution**: In the sentence "The trophy didn't fit in the suitcase because it was too big", visualize attention to see if any head correctly links "it" to "trophy".

2. **Layer depth**: Compare attention patterns in layer 0 vs layer 11 for the sentence "Apple stock rose after the iPhone launch". Do deeper layers show more semantic attention?

3. **Model size**: Load `bert-large-uncased` (if you have the RAM) and compare its parameter distribution to bert-base. How does the ratio of attention to FFN parameters change?

4. **Causal vs bidirectional**: Run the same sentence through both BERT and DistilGPT2. Compare the first token's attention pattern in each. Why is BERT's [CLS] token special?

## Key Takeaways

- **Self-attention** computes a weighted sum of all token representations using Query, Key, Value projections scaled by $\sqrt{d_k}$
- **Multi-head attention** runs multiple attention patterns in parallel, each learning different types of relationships (syntactic, positional, semantic)
- **Three architectures**: Encoder-only (BERT, bidirectional), Decoder-only (GPT-2, causal), Encoder-Decoder (T5, cross-attention)
- **Contextual embeddings**: Unlike static word vectors, Transformer representations change based on context — the same word gets different representations in different sentences
- **Feed-forward networks** contain the majority of a Transformer's parameters and encode learned knowledge

## Next Steps & Resources

**Next notebook**: [00_03 — HuggingFace Ecosystem Tour](00_03_huggingface_ecosystem.ipynb) — learn how to navigate the HuggingFace Hub, use AutoClasses, and work with pipelines.

**Resources:**
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) — The original Transformer paper
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/) — Best visual guide
- [BERT Paper](https://arxiv.org/abs/1810.04805) — Encoder-only architecture
- [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) — Decoder-only architecture
- [HuggingFace Model Documentation](https://huggingface.co/docs/transformers/model_doc/bert) — Detailed BERT internals